In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# === Load data ===
file_path = "Figure_7A_TSC22D2_Phosphomutants_Ver2.csv"
phospho_df = pd.read_csv(file_path)
wt_seq = "".join(phospho_df['Residue'].astype(str))
mutants = ['7A', '16A', '27A', '40A']

# === Build mutant sequences ===
mutant_seqs = {}
for mut in mutants:
    seq_list = []
    for _, row in phospho_df.iterrows():
        if row[mut] == "WT" or row[mut] == "-":
            seq_list.append(row['Residue'])
        else:
            seq_list.append(row[mut].split("->")[-1])
    mutant_seqs[mut] = "".join(seq_list)

# === Charge calculation: Biopython if available ===
try:
    from Bio.SeqUtils.ProtParam import ProteinAnalysis
    def per_residue_charge(seq, pH=7.4):
        charges = []
        for i in range(len(seq)):
            subseq = seq[:i+1]
            analysis = ProteinAnalysis(subseq)
            charge_now = analysis.charge_at_pH(pH)
            charge_prev = ProteinAnalysis(subseq[:-1]).charge_at_pH(pH) if i > 0 else 0
            charges.append(charge_now - charge_prev)
        return charges
except ImportError:
    # Fallback simple charge model
    aa_charge = {'D': -1, 'E': -1, 'K': +1, 'R': +1, 'H': 0.1}
    def per_residue_charge(seq, pH=7.4):
        return [aa_charge.get(aa, 0) for aa in seq]

# === Compute charges ===
wt_charge = per_residue_charge(wt_seq)
mutant_charges = {mut: per_residue_charge(seq) for mut, seq in mutant_seqs.items()}

# === Add phosphorylation charge to WT and mutants ===
for i, row in phospho_df.iterrows():
    if row['Type'] == 'Phospho' and row['Residue'] in ['S', 'T']:
        wt_charge[i] += -2
for mut in mutants:
    for i, row in phospho_df.iterrows():
        if row['Type'] == 'Phospho' and row['Residue'] in ['S', 'T']:
            if row[mut] in ['WT', '-']:
                mutant_charges[mut][i] += -2

# === Plot line graphs for WT vs Mutants ===
fig, axes = plt.subplots(len(mutants), 1, figsize=(6, 4), sharex=True)
phospho_only = phospho_df[phospho_df['Type'] == 'Phospho']

for i, mut in enumerate(mutants):
    ax = axes[i]
    # WT (thick light gray)
    ax.plot(range(1, len(wt_charge)+1), wt_charge, color='#6fd8c4', linestyle='-', linewidth=.5, alpha=1, label='WT')
    # Mutant (red)
    ax.plot(range(1, len(mutant_charges[mut])+1), mutant_charges[mut], color='#e8e8e8', linewidth=.5, alpha=1, label=mut)
    
    # Mark phosphosites at y=0
    for _, row in phospho_only.iterrows():
        pos = int(row['Position'])
        color = 'black' if row[mut] in ['WT','-'] else 'red'
        ax.plot(pos, 0, 'o', color=color, markersize=1, markeredgecolor='lightgray')
    
    ax.set_ylabel(mut, rotation=0, labelpad=30, fontsize=12)
    ax.set_xlim(0, 650)

axes[-1].set_xlabel("Residue Number", fontsize=12)
fig.suptitle("Net Charge per Residue (WT vs Mutants) with Phosphorylation Sites", fontsize=16)
plt.tight_layout()
plt.savefig("Figure_7A_TSC22D2_Phosphomutants_Net-charge-per-residue.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# === Load data ===
file_path = "Figure_7A_TSC22D2_Phosphomutants_Ver2.csv"
phospho_df = pd.read_csv(file_path)
wt_seq = "".join(phospho_df['Residue'].astype(str))
mutants = ['7A', '16A', '27A', '40A']

# === Build mutant sequences ===
mutant_seqs = {}
for mut in mutants:
    seq_list = []
    for _, row in phospho_df.iterrows():
        if row[mut] == "WT" or row[mut] == "-":
            seq_list.append(row['Residue'])
        else:
            seq_list.append(row[mut].split("->")[-1])
    mutant_seqs[mut] = "".join(seq_list)

# === Charge calculation ===
try:
    from Bio.SeqUtils.ProtParam import ProteinAnalysis

    def per_residue_charge(seq, pH=7.4):
        charges = []
        for i in range(len(seq)):
            subseq = seq[:i+1]
            analysis = ProteinAnalysis(subseq)
            charge_now = analysis.charge_at_pH(pH)
            charge_prev = ProteinAnalysis(subseq[:-1]).charge_at_pH(pH) if i > 0 else 0
            charges.append(charge_now - charge_prev)
        return charges

except ImportError:
    aa_charge = {'D': -1, 'E': -1, 'K': +1, 'R': +1, 'H': 0.1}

    def per_residue_charge(seq, pH=7.4):
        return [aa_charge.get(aa, 0) for aa in seq]

# === Compute charges ===
wt_charge = per_residue_charge(wt_seq)
mutant_charges = {mut: per_residue_charge(seq) for mut, seq in mutant_seqs.items()}

# === Add phosphorylation charge ===
for i, row in phospho_df.iterrows():
    if row['Type'] == 'Phospho' and row['Residue'] in ['S', 'T']:
        wt_charge[i] += -2

for mut in mutants:
    for i, row in phospho_df.iterrows():
        if row['Type'] == 'Phospho' and row['Residue'] in ['S', 'T']:
            if row[mut] in ['WT', '-']:
                mutant_charges[mut][i] += -2

# === Create unified y-limit across WT + mutants ===
all_charges = wt_charge + sum([mutant_charges[m] for m in mutants], [])
ymin, ymax = min(all_charges), max(all_charges)

# === Create 5 independent subplots (WT + 4 mutants) ===
fig, axes = plt.subplots(1 + len(mutants), 1, figsize=(6, 7), sharex=True)

phospho_only = phospho_df[phospho_df['Type'] == 'Phospho']

# === 1) WT only ===
ax0 = axes[0]
ax0.plot(range(1, len(wt_charge)+1), wt_charge,
         color='#6fd8c4', linewidth=0.5)
for _, row in phospho_only.iterrows():
    pos = int(row['Position'])
    ax0.plot(pos, 0, 'o', color='black', markersize=1)
ax0.set_ylabel("WT", rotation=0, labelpad=30, fontsize=12)
ax0.set_xlim(0, 650)
ax0.set_ylim(ymin, ymax)

# === 2–5) Mutants Only ===
for i, mut in enumerate(mutants):
    ax = axes[i+1]
    ax.plot(range(1, len(mutant_charges[mut])+1), mutant_charges[mut],
            color='#e8e8e8', linewidth=0.5)

    for _, row in phospho_only.iterrows():
        pos = int(row['Position'])
        color = 'black' if row[mut] in ['WT', '-'] else 'red'
        ax.plot(pos, 0, 'o', color=color, markersize=1)

    ax.set_ylabel(mut, rotation=0, labelpad=30, fontsize=12)
    ax.set_xlim(0, 650)
    ax.set_ylim(ymin, ymax)

axes[-1].set_xlabel("Residue Number", fontsize=12)
fig.suptitle("Net Charge per Residue (WT and Mutants Separately)", fontsize=14)
plt.tight_layout()

plt.savefig("Figure_7A_E:/TSC22D2_Phosphomutants_Net-charge-separate.pdf",
            dpi=300, bbox_inches="tight")
plt.show()
